# SIP Preprocessing — Minimal

**Stages**

1. Render PDF as RGB
2. Drop colored markup (handwritten red/blue/green annotations)
3. Sauvola binarize
4. Crop title blocks and tables off the sides
5. Save preprocessing outputs
6. Run OCR (PaddleOCR) on the cleaned diagram region
7. Classify recognized text into useful categories
8. Detect tracks, anchored on text labels (`UP MAIN`, `213T`, etc.)

Tuned for running locally with reasonable RAM (8 GB+ at 300 DPI).


## Setup

```
pip install pymupdf opencv-python scikit-image numpy pillow paddleocr paddlepaddle
```

In [ ]:
from pathlib import Path

import json

import cv2
import fitz  # PyMuPDF
import numpy as np
from IPython.display import Image as IPyImage
from PIL import Image
from skimage.filters import threshold_sauvola

Image.MAX_IMAGE_PIXELS = None  # engineering scans are huge — bypass Pillow's bomb guard


# ---- Edit these ------------------------------------------------------------
PDF_PATH = Path("./MUNDEWADI-Signaling_Interlocking_Plan__SIP__from_Railways.pdf")
TARGET_DPI = 300
OUT_DIR = Path("./sip_outputs")

# The Sauvola window must scale with DPI: it should span ~5x the typical stroke
# width. At 300 DPI strokes are ~10 px so a 51 px window is right; at 150 DPI
# strokes are ~5 px and a 25 px window is right. The formula below picks
# automatically.
SAUVOLA_WINDOW = (TARGET_DPI // 6) | 1   # | 1 forces odd
SAUVOLA_K = 0.4

## 1. Load the PDF as RGB

PyMuPDF renders the page directly at the target DPI. No intermediate
full-resolution raster.

In [ ]:
doc = fitz.open(PDF_PATH)
page = doc[0]
zoom = TARGET_DPI / 72
pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)
rgb = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
if pix.n == 4:
    rgb = rgb[:, :, :3].copy()
doc.close()
print(f"Loaded {rgb.shape} at {TARGET_DPI} DPI")

## 2. Drop colored markup

Build a grayscale where every pixel with non-trivial chroma (red/blue/green
handwritten markup) is forced to paper-white. Black structural ink is
unchanged. This is the only step we need from the elaborate "color
separation" — one filter, two lines.

In [ ]:
rgb_max = rgb.max(axis=2)
rgb_min = rgb.min(axis=2)
chroma = cv2.subtract(rgb_max, rgb_min)   # uint8, saturating

gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
gray[chroma > 20] = 255   # blank out anything chromatic
del rgb_max, rgb_min, chroma

## 3. Binarize (Sauvola)

Adaptive thresholding. Sauvola handles uneven illumination and stroke-width
variation naturally, so we don't need a separate background-normalization
or contrast-enhancement step.

In [ ]:
thr = threshold_sauvola(gray, window_size=SAUVOLA_WINDOW, k=SAUVOLA_K)
binary = ((gray < thr).astype(np.uint8)) * 255
print(f"Window: {SAUVOLA_WINDOW}, foreground: {(binary > 0).mean()*100:.2f}%")
del thr

## 4. Crop tables and title blocks off the sides

Title blocks and edge legends fill columns with **much higher ink density**
than the diagram's track region. We can detect those high-density edge bands
and crop them off:

1. Compute a smoothed column-ink-density profile.
2. Threshold against `high_density_factor × median(diagram-region density)`.
3. Morphologically close small gaps in the high-density mask (table grids
   contain narrow low-density rows between borders that we don't want to
   stop at).
4. Walk inward from each edge through any high-density region.

Tune `high_density_factor` if the crop is too aggressive (raise it, e.g.,
2.5) or too loose (lower it, e.g., 1.5).

In [ ]:
def crop_to_diagram(
    binary,
    smooth_window=200,
    high_density_factor=2.0,
    bridge_gap=2000,
    edge_margin=200,
):
    h, w = binary.shape
    col_ink = (binary > 0).sum(axis=0).astype(float)
    # Smooth with edge-padding so right/left columns aren't artificially low.
    pad = smooth_window // 2
    padded = np.pad(col_ink, pad, mode="edge")
    smooth = np.convolve(padded, np.ones(smooth_window) / smooth_window, mode="valid")[:w]

    # Threshold relative to the median density of the central diagram region.
    diagram_median = np.median(smooth[w // 4 : 3 * w // 4])
    is_high = (smooth >= diagram_median * high_density_factor).astype(np.uint8)

    # Bridge small gaps inside table grids so the inward walk doesn't stop early.
    kernel = np.ones((1, max(3, bridge_gap | 1)), np.uint8)
    closed = cv2.morphologyEx(is_high.reshape(1, -1) * 255, cv2.MORPH_CLOSE, kernel).ravel() > 0

    x_min = 0
    if closed[0]:
        while x_min < w - 1 and closed[x_min]:
            x_min += 1
    x_max = w - 1
    if closed[-1]:
        while x_max > 0 and closed[x_max]:
            x_max -= 1

    return max(0, x_min - edge_margin), min(w, x_max + edge_margin)


x_min, x_max = crop_to_diagram(binary)
cropped = binary[:, x_min:x_max]
print(f"Crop: x=[{x_min}, {x_max}]  ({cropped.shape[1]} px / {binary.shape[1]} px = "
      f"{cropped.shape[1]/binary.shape[1]*100:.1f}%)")

## 5. Save and inspect

In [ ]:
OUT_DIR.mkdir(exist_ok=True)
binary_path = OUT_DIR / "binary.png"
cv2.imwrite(str(binary_path), cropped)

# Save a downsampled preview for inline display. The full binary at 300 DPI
# is too wide to render usefully inline anyway.
preview_path = OUT_DIR / "binary_preview.png"
preview = cv2.resize(
    cropped,
    (3000, int(cropped.shape[0] * 3000 / cropped.shape[1])),
    interpolation=cv2.INTER_AREA,
)
cv2.imwrite(str(preview_path), preview)

print(f"Full binary : {binary_path}  ({cropped.shape[1]} x {cropped.shape[0]})")
print(f"Preview     : {preview_path}  ({preview.shape[1]} x {preview.shape[0]})")
IPyImage(str(preview_path))

## 6. Text detection and OCR

Detect every text region in the diagram and extract its content. This produces
a JSON of `{text, bbox, score}` entries — signal IDs (`S63`), track-circuit IDs
(`116AT`), station names (`MUNDHEWADI`, `SOLAPUR`), KM markers (`KM 429`),
and track labels (`UP MAIN`, `DN LOOP`).

We use **PaddleOCR** with:
- `use_textline_orientation=True` — handles vertical text (KM markers) and rotated labels.
- The cleaned grayscale (markup removed) as input — better than the binary
  because adaptive thresholding can fragment thin character strokes.
- The same horizontal crop as the binary, so coordinates align with downstream
  stages.
- Tile-based processing — OCR detection runs better on tiles than one giant image.

**First-run note**: PaddleOCR downloads its detection and recognition models
the first time you initialize it (a few hundred MB). This is a one-time download
and gets cached under `~/.paddleocr/`. Subsequent runs use the cache.

CPU runtime is ~30-60s per SIP. No GPU required.

In [ ]:
# Apply the same crop used for the binary to the grayscale input
gray_cropped = gray[:, x_min:x_max]
print(f"OCR input shape: {gray_cropped.shape}")

In [ ]:
# This cell takes ~10-30s on first run (model download), faster afterwards.
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
    lang='en',
    text_rec_score_thresh=0.5,  # drop low-confidence garbage like 'WoooL'
)
print("PaddleOCR ready")

In [ ]:
import re

# Tile-based OCR. Detection works better on moderate-size tiles than one
# giant image, and lets us bound memory.
TILE = 2000
OVERLAP = 200

# OCR wants 3-channel input
ocr_input = cv2.cvtColor(gray_cropped, cv2.COLOR_GRAY2RGB)
H, W = ocr_input.shape[:2]


def normalize_text(s):
    """Collapse common OCR artifacts so 'S 19' == 'S19'.

    For short purely-alphanumeric tokens (signal IDs, track-circuit IDs, point
    IDs), strip internal whitespace. Leave longer text (multi-word labels)
    alone so we don't mangle 'UP MAIN' into 'UPMAIN'.
    """
    s = s.strip()
    if len(s) <= 8 and re.match(r'^[A-Z0-9./\s]+$', s, re.IGNORECASE):
        s = re.sub(r'\s+', '', s)
    return s


text_entities = []
for y0 in range(0, H, TILE - OVERLAP):
    for x0 in range(0, W, TILE - OVERLAP):
        y1 = min(y0 + TILE, H)
        x1 = min(x0 + TILE, W)
        tile = ocr_input[y0:y1, x0:x1]
        if tile.size == 0:
            continue
        results = ocr.predict(tile)
        for r in results:
            for poly, text, score in zip(r['rec_polys'], r['rec_texts'], r['rec_scores']):
                xs = [int(p[0]) + x0 for p in poly]
                ys = [int(p[1]) + y0 for p in poly]
                bbox = [min(xs), min(ys), max(xs) - min(xs), max(ys) - min(ys)]
                text_entities.append({
                    "text": text,
                    "text_normalized": normalize_text(text),
                    "score": float(score),
                    "bbox": bbox,
                })

# Deduplicate: tile overlap can produce near-duplicate detections.
# Match on normalized text so 'S 19' (in tile A) and 'S19' (in tile B) collapse.
def dedupe(entities, iou_thr=0.3):
    def iou(a, b):
        ax2, ay2 = a[0] + a[2], a[1] + a[3]
        bx2, by2 = b[0] + b[2], b[1] + b[3]
        ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
        ix2, iy2 = min(ax2, bx2), min(ay2, by2)
        if ix2 <= ix1 or iy2 <= iy1: return 0.0
        inter = (ix2 - ix1) * (iy2 - iy1)
        union = a[2] * a[3] + b[2] * b[3] - inter
        return inter / union if union else 0.0
    entities = sorted(entities, key=lambda e: -e["score"])
    kept = []
    for e in entities:
        if not any(iou(e["bbox"], k["bbox"]) > iou_thr
                   and e["text_normalized"] == k["text_normalized"]
                   for k in kept):
            kept.append(e)
    return kept

text_entities = dedupe(text_entities)
print(f"Detected {len(text_entities)} unique text regions")

In [ ]:
# Save JSON and an annotated overlay
text_json_path = OUT_DIR / "text.json"
with open(text_json_path, "w") as f:
    json.dump(text_entities, f, indent=2)

overlay = cv2.cvtColor(gray_cropped, cv2.COLOR_GRAY2BGR)
for t in text_entities:
    x, y, w, h = t["bbox"]
    cv2.rectangle(overlay, (x, y), (x + w, y + h), (0, 200, 255), 2)
    # Tiny label (only readable at full res; that's fine)
    cv2.putText(overlay, t["text"][:20], (x, max(15, y - 3)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 200, 255), 1, cv2.LINE_AA)

text_overlay_path = OUT_DIR / "text_overlay.png"
cv2.imwrite(str(text_overlay_path), overlay)

# Inline preview
preview3 = cv2.resize(
    overlay,
    (3000, int(overlay.shape[0] * 3000 / overlay.shape[1])),
    interpolation=cv2.INTER_AREA,
)
preview3_path = OUT_DIR / "text_overlay_preview.png"
cv2.imwrite(str(preview3_path), preview3)

print(f"JSON    : {text_json_path}")
print(f"Overlay : {text_overlay_path}")
IPyImage(str(preview3_path))

In [ ]:
# Sample of what was found, sorted by Y then X (reading order)
sample = sorted(text_entities, key=lambda t: (t["bbox"][1], t["bbox"][0]))
print(f"Found {len(sample)} text regions. First 30:")
for t in sample[:30]:
    print(f"  '{t['text']:30s}' score={t['score']:.2f} bbox={t['bbox']}")

## 7. Classify text into categories

Bucket recognized text using regex on the **normalized** strings.

| Category         | Matches                                            | Examples                       |
|------------------|----------------------------------------------------|--------------------------------|
| `signal_id`      | letter prefix + digits, optional letter suffix     | `S63`, `SH6`, `C02`, `S2A`     |
| `track_circuit`  | digits + optional `A` + `T`                        | `116AT`, `305T`, `217T`        |
| `km_marker`      | `KM` followed by digits                            | `KM 429`, `KM.430.330m`        |
| `track_label`    | UP/DN/DOWN + MAIN/LOOP/etc, ≤4 words               | `UP MAIN`, `DN LOOP`           |
| `point_id`       | 100-199, optional `/123` partner                   | `103`, `113/114`               |
| `dimension`      | digits ± decimal, optional `m` suffix              | `180m`, `1000m`, `709.5m`      |
| `note`           | everything else                                    |                                |

Note: `dimension` covers things that look like distances/lengths. `point_id`
is restricted to the 100-range to avoid eating dimensions like `200` or `850`
that happen to be 3 digits. We're trading a few real false-positive station
names for clean buckets — station-name detection is better done by position
later (leftmost/rightmost text in the diagram band), not by text patterns.

In [ ]:
PATTERNS = {
    "signal_id":     re.compile(r"^(S|SH|C|CO|H|D|AS|IB|GL|BSLB)\d{1,3}[A-Z]?$", re.IGNORECASE),
    "track_circuit": re.compile(r"^\d{1,4}A?T$"),
    "km_marker":     re.compile(r"^J?K\.?\s?M\.?\s?\d{2,4}", re.IGNORECASE),
    "track_label":   re.compile(
        r"^(UP|DN|DOWN)\.?\s*(MAIN|LOOP|LINE|SIDING|YARD|SHUNT)"
        r"(\s+(HOME|STARTER|ADV\.?\s*STARTER|DIST\.?))?\s*\.?$",
        re.IGNORECASE,
    ),
    "point_id":      re.compile(r"^1[01]\d(\s?/\s?1[01]\d)?$"),
    "dimension":     re.compile(r"^\d{1,5}(\.\d+)?\s*m\.?$", re.IGNORECASE),
}


def classify_text(text):
    """Return the first matching category, or 'note' if nothing matches."""
    s = text.strip()
    for cat, pat in PATTERNS.items():
        if pat.search(s):
            return cat
    return "note"


for t in text_entities:
    t["category"] = classify_text(t["text_normalized"])

from collections import Counter
counts = Counter(t["category"] for t in text_entities)
print("Category breakdown:")
for cat, n in counts.most_common():
    print(f"  {cat:15s}: {n}")

print()
for cat in counts:
    print(f"\n{cat}:")
    for t in [e for e in text_entities if e['category'] == cat][:8]:
        print(f"  '{t['text_normalized']:30s}' bbox={t['bbox']}")

In [ ]:
# Resave with category included
with open(OUT_DIR / "text.json", "w") as f:
    json.dump(text_entities, f, indent=2)

# Color-coded overlay
CATEGORY_COLORS = {
    "signal_id":     (255, 100, 100),  # red
    "track_circuit": (100, 255, 100),  # green
    "km_marker":     (255, 255, 100),  # yellow
    "track_label":   (100, 200, 255),  # cyan
    "dimension":     (255, 100, 255),  # magenta
    "point_id":      (200, 200, 200),  # light gray
    "note":          (80, 80, 80),     # dark gray
}

overlay = cv2.cvtColor(gray_cropped, cv2.COLOR_GRAY2BGR)
for t in text_entities:
    x, y, w, h = t["bbox"]
    color = CATEGORY_COLORS.get(t["category"], (200, 200, 200))
    cv2.rectangle(overlay, (x, y), (x + w, y + h), color, 2)

cv2.imwrite(str(OUT_DIR / "text_categorized_overlay.png"), overlay)

preview4 = cv2.resize(
    overlay,
    (3000, int(overlay.shape[0] * 3000 / overlay.shape[1])),
    interpolation=cv2.INTER_AREA,
)
cv2.imwrite(str(OUT_DIR / "text_categorized_preview.png"), preview4)
IPyImage(str(OUT_DIR / "text_categorized_preview.png"))

## 8. Track detection (text-anchored)

Earlier attempts at finding tracks from geometry alone kept getting confused
by platform borders, table grids, and dimension lines. Now that we have OCR
results, we can flip the approach: **use the text labels as anchors**.

A track is "real" if it has a `track_label` text (`UP MAIN`, `DN LOOP`) or a
`track_circuit` ID (`213T`, `241T`) sitting next to it. Platforms and table
borders won't have those labels.

Algorithm:

1. Build a horizontal-line skeleton from the cropped binary (same as before:
   morphological opening with a horizontal kernel + skeletonize).
2. Connected components of the skeleton are candidate track segments.
3. For each `track_label` and `track_circuit` text, find the nearest
   skeleton component within a vertical search range. That component "owns"
   the text as an anchor.
4. Components with at least one anchor → real tracks. Drop the rest.
5. Trace each track component as a polyline (longest path through the
   skeleton, simplified with Douglas-Peucker).
6. Name each track from its anchors: prefer a `track_label` if present, else
   list its `track_circuit` IDs.

In [ ]:
from collections import deque
from skimage.morphology import skeletonize


def build_track_skeleton(binary, horiz_kernel_len=60, min_length=200):
    """Skeletonize horizontal line structures and label connected components.

    No gap-bridging — earlier attempts at bridging gaps between skeleton
    fragments tended to fuse parallel tracks into one mega-component, which
    confused the anchor-matching logic. Instead, we accept skeleton
    fragmentation and let each text anchor pick its own component.
    """
    horiz_kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (horiz_kernel_len, 1))
    horiz_only = cv2.morphologyEx(binary, cv2.MORPH_OPEN, horiz_kernel)
    skel = skeletonize(horiz_only > 0).astype(np.uint8)
    num, labels, stats, _ = cv2.connectedComponentsWithStats(skel, connectivity=8)
    keep_ids = [i for i in range(1, num) if stats[i, cv2.CC_STAT_AREA] >= min_length]
    return labels, stats, keep_ids


def anchor_text_to_component(text_bbox, labels, stats, keep_ids,
                             max_y_distance=300):
    """Find the skeleton component whose horizontal extent contains the text's
    center X and is closest in Y. Falls back to nearest by Y if no
    horizontally-containing component exists.
    """
    tx, ty, tw, th = text_bbox
    text_cx = tx + tw // 2
    text_cy = ty + th // 2

    containing = []
    near = []
    for cid in keep_ids:
        cx = stats[cid, cv2.CC_STAT_LEFT]
        cy_top = stats[cid, cv2.CC_STAT_TOP]
        cw = stats[cid, cv2.CC_STAT_WIDTH]
        ch = stats[cid, cv2.CC_STAT_HEIGHT]
        line_cy = cy_top + ch // 2
        dy = abs(line_cy - text_cy)
        if dy > max_y_distance:
            continue
        if cx <= text_cx <= cx + cw:
            containing.append((dy, cid))
        else:
            ov = max(0, min(cx + cw, tx + tw) - max(cx, tx))
            if ov >= tw * 0.3:
                near.append((dy, cid))

    if containing:
        return min(containing)[1]
    if near:
        return min(near)[1]
    return None


def trace_polyline(component_pixels):
    """Trace longest path through skeleton component pixels (BFS twice)."""
    pixels = set(component_pixels)
    if not pixels:
        return []
    def neighbors(p):
        x, y = p
        return [(x+dx, y+dy) for dx in (-1,0,1) for dy in (-1,0,1)
                if (dx, dy) != (0, 0) and (x+dx, y+dy) in pixels]
    endpoints = [p for p in pixels if len(neighbors(p)) == 1]
    start = min(endpoints, key=lambda p: p[0]) if endpoints else min(pixels, key=lambda p: p[0])
    parent = {start: None}; queue = deque([start])
    farthest, max_d = start, 0; dist = {start: 0}
    while queue:
        u = queue.popleft()
        for v in neighbors(u):
            if v not in parent:
                parent[v] = u; dist[v] = dist[u] + 1
                if dist[v] > max_d: max_d, farthest = dist[v], v
                queue.append(v)
    path = []
    cur = farthest
    while cur is not None:
        path.append(cur); cur = parent[cur]
    path.reverse()
    return path


labels, stats, keep_ids = build_track_skeleton(cropped)
print(f"Skeleton: {len(keep_ids)} candidate components")

# Anchor each text to a component. We'll build tracks per-anchor, not per-
# component, so a single physical line that's been split into 3 skeleton
# fragments becomes 3 JSON entries — but they share a name (and color) if
# they're labeled the same, so visually they read as one track.
anchored = []  # list of (text_entity, component_id)
for t in text_entities:
    if t["category"] not in ("track_label", "track_circuit"):
        continue
    # track_circuit IDs sit ON the line — tight Y window.
    # track_label text (UP MAIN, DN LOOP) can be above or below — wider.
    max_y = 80 if t["category"] == "track_circuit" else 300
    cid = anchor_text_to_component(t["bbox"], labels, stats, keep_ids,
                                   max_y_distance=max_y)
    if cid is not None:
        anchored.append((t, cid))

print(f"Anchored {len(anchored)} text entities to components")

In [ ]:
# One track per anchor. Multiple anchors on the same component => multiple
# track entries (intentional — preserves all label evidence).
TRACK_POLYLINE_EPSILON = 2.0

tracks = []
seen_component_for_label = {}  # (text_normalized, cid) -> already emitted, dedupe

for text_entity, cid in anchored:
    key = (text_entity["text_normalized"], cid)
    if key in seen_component_for_label:
        # Same label already produced a track for this exact component.
        # Merge the secondary anchor into the existing track's anchors list.
        seen_component_for_label[key]["anchors"].append(text_entity["text_normalized"])
        continue

    ys, xs = np.where(labels == cid)
    component_pixels = list(zip(xs.tolist(), ys.tolist()))
    raw_path = trace_polyline(component_pixels)
    if len(raw_path) < 2:
        continue
    contour = np.array(raw_path, dtype=np.int32).reshape(-1, 1, 2)
    simp = cv2.approxPolyDP(contour, TRACK_POLYLINE_EPSILON, False)
    polyline = [[int(p[0][0]), int(p[0][1])] for p in simp]

    track = {
        "id": f"track_{len(tracks)+1:03d}",
        "type": "track",
        "name": text_entity["text_normalized"],
        "anchor_category": text_entity["category"],
        "polyline": polyline,
        "anchors": [text_entity["text_normalized"]],
    }
    tracks.append(track)
    seen_component_for_label[key] = track

# For tracks named by track_circuit, prefer a track_label name if any other
# track_label anchor sits on the same component. Visit all label anchors
# and propagate their name to circuit-named tracks on the same component.
component_to_label_name = {}
for text_entity, cid in anchored:
    if text_entity["category"] == "track_label":
        component_to_label_name.setdefault(cid, text_entity["text_normalized"])

for t in tracks:
    if t["anchor_category"] == "track_circuit":
        # Find which component this track came from (via its first polyline pt)
        x0, y0 = t["polyline"][0]
        cid = labels[y0, x0]
        if cid in component_to_label_name:
            t["name"] = component_to_label_name[cid]

print(f"Built {len(tracks)} track entries")
for t in tracks:
    print(f"  {t['id']}: name={t['name']!r:25s} pts={len(t['polyline']):3d}  "
          f"anchors={t['anchors']}")

In [ ]:
# Save JSON and overlay. Color by track NAME so all entries that represent
# the same logical track get the same color.
import hashlib

def color_for_name(name):
    if not name:
        return (160, 160, 160)
    h = int(hashlib.md5(name.encode()).hexdigest()[:6], 16)
    b = 80 + (h & 0xFF) % 160
    g = 80 + ((h >> 8) & 0xFF) % 160
    r = 80 + ((h >> 16) & 0xFF) % 160
    return (b, g, r)


with open(OUT_DIR / "tracks.json", "w") as f:
    json.dump(tracks, f, indent=2)

overlay = cv2.cvtColor(cropped, cv2.COLOR_GRAY2BGR)
for t in tracks:
    color = color_for_name(t["name"])
    pts = np.array(t["polyline"], dtype=np.int32)
    cv2.polylines(overlay, [pts], False, color, 4)
    if t["name"]:
        x, y = t["polyline"][0]
        cv2.putText(overlay, t["name"], (x + 10, max(20, y - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2, cv2.LINE_AA)

cv2.imwrite(str(OUT_DIR / "tracks_overlay.png"), overlay)
preview5 = cv2.resize(
    overlay,
    (3000, int(overlay.shape[0] * 3000 / overlay.shape[1])),
    interpolation=cv2.INTER_AREA,
)
cv2.imwrite(str(OUT_DIR / "tracks_overlay_preview.png"), preview5)
IPyImage(str(OUT_DIR / "tracks_overlay_preview.png"))

## Optional: things you might add

- **Deskew** — only matters for SIPs with visible rotation. Use `deskew`
  (PyPI) or fit a line to long horizontal segments via `cv2.HoughLinesP`.
- **Despeckle** — drop connected components below ~5 px:
  ```python
  num, labels, stats, _ = cv2.connectedComponentsWithStats(cropped)
  keep = stats[:, 4] >= 5
  keep[0] = False
  cropped = (keep[labels].astype(np.uint8)) * 255
  ```
- **Vertical crop** — same trick as horizontal, with a `(1, kernel_len)`
  kernel, if your SIPs have header/footer bands worth removing.
- **VLM fallback for low-confidence OCR** — re-recognize crops with PaddleOCR
  confidence below ~0.6 using Qwen2.5-VL on Colab GPU. Should fix ~`WoooL`-type
  garbage that survives the score threshold.
- **Turnout detection** — places where two anchored tracks share skeleton
  pixels are turnouts. Each turnout gets `{id, vertex: [vx, vy], connects: [...]}`.
